# Section 6 — Order and Rate of Convergence

[← Newton's Method](05_newton_method.ipynb) · [Module 1](README.md)

There are two related uses of the phrase **rate/order of convergence** in this
module:

1. Big-$O$ descriptions of sequences or functions approaching a limit;
2. the order $\alpha$ of an iterative numerical method.

Both describe *how fast an error goes to zero*.

### Learning goals

- identify Big-$O$ rates for sequences and functions;
- estimate iterative convergence order from error data;
- estimate an asymptotic error constant;
- connect $g'(p)$ to linear fixed-point convergence;
- observe Newton's quadratic convergence near a simple root.


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np


## 1. Big-$O$ for sequences

Suppose $a_n\to0$. We write

$$
a_n=O(n^{-q})
$$

if, for sufficiently large $n$,

$$
|a_n|\le \frac{C}{n^q}
$$

for some constant $C$.

For the review sequence

$$
a_n=\frac{n^2+2n+5}{n^4},
$$

divide by the dominant power:

$$
a_n=
\frac{1}{n^2}
+\frac{2}{n^3}
+\frac{5}{n^4}.
$$

Therefore

$$
a_n=O\left(\frac{1}{n^2}\right).
$$


In [ ]:
n_values = np.array([10, 20, 40, 80, 160], dtype=float)
sequence = (
    n_values**2 + 2.0 * n_values + 5.0
) / n_values**4

print("   n          a_n          n^2 a_n")
for n_value, a_value in zip(n_values, sequence):
    print(
        f"{n_value:5.0f}   {a_value:11.4e}   "
        f"{n_value**2 * a_value:10.6f}"
    )


## 2. Big-$O$ for a function approaching a limit

The review also considers

$$
\frac{\sin h}{h}\to1
\qquad\text{as }h\to0.
$$

Taylor expansion gives

$$
\frac{\sin h}{h}
=
1-\frac{h^2}{6}+O(h^4).
$$

Thus

$$
\left|\frac{\sin h}{h}-1\right|=O(h^2).
$$

The *error* is second order in $h$.


In [ ]:
h_values = 2.0 ** (-np.arange(2, 12))
errors_h = np.abs(np.sin(h_values) / h_values - 1.0)

print("      h           error         error / h^2")
for h_value, error in zip(h_values, errors_h):
    print(
        f"{h_value:9.6f}   {error:11.4e}   "
        f"{error / h_value**2:11.6f}"
    )


## 3. Iterative convergence

Suppose $p_n\to p$ and define

$$
e_n=|p_n-p|.
$$

The sequence has order $\alpha$ and asymptotic error constant $\lambda$ if

$$
\lim_{n\to\infty}
\frac{|p_{n+1}-p|}
{|p_n-p|^\alpha}
=
\lambda,
\qquad
0<\lambda<\infty.
$$

Important cases:

- $\alpha=1$: linear convergence;
- $\alpha=2$: quadratic convergence.


### Fixed-point iteration: linear convergence

For

$$
g(x)=\frac{5}{x+4},
\qquad p=1,
$$

we have

$$
g'(1)=-\frac15.
$$

We therefore expect linear convergence with

$$
\lambda=|g'(1)|=\frac15.
$$


In [ ]:
def g(x):
    return 5.0 / (x + 4.0)


p = 1.0
values = [1.5]

for _ in range(10):
    values.append(g(values[-1]))

errors = [abs(value - p) for value in values]

print(" n       error e_n       e_n / e_(n-1)")
for n in range(1, len(errors)):
    ratio = errors[n] / errors[n - 1]
    print(f"{n:2d}   {errors[n]:12.5e}   {ratio:12.8f}")


### Newton's method: quadratic convergence

For a simple root and suitable hypotheses, Newton's method is locally quadratic.

For

$$
f(x)=2-e^x,
$$

the root is $p=\ln2$.


In [ ]:
def f(x):
    return 2.0 - math.exp(x)


def derivative_f(x):
    return -math.exp(x)


p = math.log(2.0)
newton_values = [0.5]

for _ in range(6):
    p_old = newton_values[-1]
    p_new = p_old - f(p_old) / derivative_f(p_old)
    newton_values.append(p_new)

newton_errors = [
    abs(value - p) for value in newton_values
]

print(" n       error e_n       e_n / e_(n-1)^2")
for n in range(1, len(newton_errors)):
    previous_error = newton_errors[n - 1]

    if previous_error == 0:
        break

    ratio = newton_errors[n] / previous_error**2
    print(
        f"{n:2d}   {newton_errors[n]:12.5e}   "
        f"{ratio:12.8f}"
    )


## 4. Estimating observed order

With three consecutive nonzero errors,

$$
\alpha_n
\approx
\frac{
\log(e_{n+1}/e_n)
}{
\log(e_n/e_{n-1})
}.
$$


In [ ]:
def observed_orders(error_values):
    """Estimate convergence orders from positive errors."""
    orders = []

    for index in range(1, len(error_values) - 1):
        e_previous = error_values[index - 1]
        e_current = error_values[index]
        e_next = error_values[index + 1]

        if min(e_previous, e_current, e_next) <= 0:
            continue

        denominator = math.log(e_current / e_previous)

        if denominator == 0:
            continue

        order = math.log(e_next / e_current) / denominator
        orders.append((index, order))

    return orders


print("Fixed-point observed orders:")
for index, order in observed_orders(errors):
    print(f"n={index:2d}: alpha ≈ {order:.6f}")

print("\nNewton observed orders:")
for index, order in observed_orders(newton_errors):
    print(f"n={index:2d}: alpha ≈ {order:.6f}")


In [ ]:
plt.figure(figsize=(8, 5))
plt.semilogy(
    range(len(errors)),
    errors,
    marker="o",
    label="fixed-point iteration",
)
plt.semilogy(
    range(len(newton_errors)),
    newton_errors,
    marker="o",
    label="Newton's method",
)
plt.xlabel("iteration n")
plt.ylabel(r"absolute error $|p_n-p|$")
plt.title("Convergence of Two Iterative Methods")
plt.grid(alpha=0.25)
plt.legend()
plt.show()


### References

- Leon Q. Brin, *Tea Time Numerical Analysis*.
- Richard L. Burden and J. Douglas Faires, *Numerical Analysis*.
- MAD 4401 Exam 1 review.

[← Newton's Method](05_newton_method.ipynb) · [↑ Module 1](README.md)
